# AgentCar — train model NLU cho edge

Ba bước: thân PhoBERT làm quen văn phong lệnh → gắn đầu học miền xe → chưng cất xuống 4 lớp.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'KHONG CO GPU — doi runtime sang T4')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1. Đưa gói lên Colab — chọn MỘT trong ba cách

**Cách 1 (dễ nhất): kéo thả.** Bấm biểu tượng thư mục ở thanh bên trái Colab,
kéo `colab_bundle.zip` từ Finder thả vào đó. Rồi chạy ô dưới.

**Cách 2: Google Drive.** Copy zip vào Drive ở máy, rồi bỏ chú thích dòng mount.

**Cách 3: nút upload.** Chỉ dùng khi hai cách trên không tiện — ô này hay hỏng
trên Safari và khi trình duyệt chặn cookie bên thứ ba.


In [ ]:
import pathlib, zipfile

TEN = "colab_bundle.zip"
tim = [p for p in (pathlib.Path("/content")/TEN, pathlib.Path(TEN)) if p.exists()]

# Cách 2 — bỏ chú thích hai dòng này nếu để zip trên Drive
# from google.colab import drive; drive.mount("/content/drive")
# tim = [pathlib.Path("/content/drive/MyDrive")/TEN]

if not tim:
    print("chưa thấy zip — thử nút upload (cách 3)")
    from google.colab import files
    up = files.upload()
    tim = [pathlib.Path(next(iter(up)))]

zipfile.ZipFile(tim[0]).extractall("/content/work")
print("đã giải nén từ", tim[0])
for f in sorted(pathlib.Path("/content/work").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/content/work')}  {f.stat().st_size/1000:.1f} KB")


## 2. Cài thư viện và tải MASSIVE


In [ ]:
!pip -q install 'transformers>=4.44' sentencepiece
!wget -q https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz
!tar xzf amazon-massive-dataset-1.0.tar.gz 1.0/data/vi-VN.jsonl
!wc -l 1.0/data/vi-VN.jsonl


## 3. Bước 1 — thân làm quen văn phong ra lệnh tiếng Việt

Train PhoBERT trên 16.5k câu MASSIVE **với nhãn của MASSIVE**. Xong thì vứt đầu, chỉ giữ thân.


In [ ]:
!cd /content/work && python3 nlu/train.py --giai-doan massive \
    --massive /content/1.0/data/vi-VN.jsonl --epochs 5 --bs 32 --out /content/runs


## 4. Bước 2 — gắn đầu của mình, học 30 thiết bị VF9


In [ ]:
!cd /content/work && python3 nlu/train.py --giai-doan mien \
    --tu-than /content/runs/massive/than.pt --epochs 6 --bs 32 --out /content/runs


## 5. Bước 3 — chưng cất xuống trò 4 lớp (thứ đóng gói lên xe)


In [ ]:
!cd /content/work && python3 nlu/train.py --giai-doan chung-cat \
    --thay /content/runs/mien/model.pt \
    --massive /content/1.0/data/vi-VN.jsonl --epochs 10 --bs 64 --out /content/runs


## 6. Tải kết quả về


In [ ]:
!cd /content/runs && zip -qr /content/ket_qua.zip chung-cat mien/ket_qua.json
!ls -lh /content/ket_qua.zip
files.download('/content/ket_qua.zip')
